### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
WHERE o.order_status IN ('Completed', 'Shipped')
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

### Debug: check out df columns name

In [ ]:
df_order_completed_shipped.columns

### Statistics on sales qty for 2023-2024 each order

In [ ]:
df_valid = df_order_completed_shipped[df_order_completed_shipped['is_free_gift'] != True]

df_each_order_qty = (
    df_valid.groupby('order_id', as_index=False)['quantity']
    .sum()
    .rename(columns={'quantity': 'order_sales_qty'})
)

df_each_order_qty

### Statistics on sales qty for 2023-2024 each customer

In [ ]:
df_valid = df_order_completed_shipped[df_order_completed_shipped['is_free_gift'] != True]

df_each_customer_qty = (
    df_valid.groupby('customer_id', as_index=False)['quantity']
    .sum()
    .rename(columns={'quantity': 'customer_sales_qty'})
)

df_each_customer_qty

In [ ]:
# 关闭数据库连接
engine.dispose()